# 01 — Data analysis (schema, EDA, wells, target)

**Variant:** `parallel_multiwell_loader` · **Competition:** [Rogii Wellbore Geology Prediction](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction)

This notebook mirrors the depth of top public kernels (e.g. Chris Deotte *EDA Starter*, Pilkwang *Leakage-Aware Pipeline*):
file inventory, column roles, missingness, target behavior, well grouping, and log1p diagnosis — then writes phase artifacts for downstream steps.

**Trace index:** `../trace_row_index.csv` · **Artifacts:** `../artifacts/01_data_analysis/`


## 1. Setup and data root

Resolve competition CSVs from Kaggle mount or local Rogii checkout (`ROGII_ROOT/data`).


In [ ]:
"""Shared imports for Rogii TVT pipeline notebooks."""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
if not (NB_DIR / "phase_runner.py").is_file():
    NB_DIR = Path(r"/lustre/work/sweeden/agent-tracing-trace-baseline/examples/rogii/traces/preprocessing/parallel_multiwell_loader/notebooks")
VARIANT_DIR = NB_DIR.parent
ROGII_ROOT = Path("/lustre/work/sweeden/rogii")
sys.path.insert(0, str(NB_DIR))
if str(ROGII_ROOT) not in sys.path:
    sys.path.insert(0, str(ROGII_ROOT))

from phase_runner import (
    ARTIFACTS_ROOT,
    TRACE_INDEX,
    require_prior_phase,
    load_phase_manifest,
    trace_steps_for_phase,
    _resolve_path,
    _read_json,
    _load_train_predict,
)

pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8-whitegrid")

PHASE = "01_data_analysis"
print("Phase:", PHASE, "| trace steps:", len(trace_steps_for_phase(PHASE)))


In [ ]:
def find_data_dir() -> Path:
    candidates = [
        ROGII_ROOT / "data",
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
    ]
    for c in candidates:
        if c.is_dir() and any(c.glob("*train*.csv")):
            return c.resolve()
    raise FileNotFoundError("Rogii train CSV not found; set ROGII_ROOT/data")

DATA_DIR = find_data_dir()
print("DATA_DIR:", DATA_DIR)
print("CSV files:", len(list(DATA_DIR.glob("*.csv"))))


## 2. Competition task recap

Predict **True Vertical Thickness (TVT)** for hidden tail rows per horizontal well. Training CSVs include labeled `TVT`; test wells omit labels. Submission ids follow `{well_hash}_{row}` in `sample_submission.csv`.


## 3. Load train / test / sample submission


In [ ]:
tp = _load_train_predict()
train_path = tp._find_default_csv(DATA_DIR, "train")
test_path = tp._find_default_csv(DATA_DIR, "test")
sample_path = tp._find_default_csv(DATA_DIR, "sample_submission")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path, nrows=5)

print("train:", train_df.shape, "| test:", test_df.shape)
print("sample_submission columns:", list(sample_sub.columns))
display(train_df.head(3))


## 4. Schema registration (SchemaSentinel)

Lock id column and target column names to the submission envelope.


In [ ]:
from pipeline.agents import SchemaSentinel
from pipeline.nb_support import ensure_id_column

schema_agent = SchemaSentinel()
schema = schema_agent.register_schema(sample_path)
id_col = schema["id_column"]
target = schema["target_columns"][0]
train_df = tp.align_train_target_to_schema(train_df, target)
train_df = ensure_id_column(train_df, id_col)
test_df = ensure_id_column(test_df, id_col)
print(json.dumps(schema, indent=2))


## 5. Column inventory and dtypes


In [ ]:
summary = pd.DataFrame({
    "column": train_df.columns,
    "dtype": train_df.dtypes.astype(str).values,
    "missing_pct": (train_df.isna().mean() * 100).round(2).values,
    "n_unique": [train_df[c].nunique(dropna=True) for c in train_df.columns],
})
display(summary.sort_values("missing_pct", ascending=False))


## 6. Missingness visualization

High missingness in `GR` and `TVT_input` is common; downstream ColumnTransformer must handle NaN sentinels.


In [ ]:
miss = train_df.isna().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
miss.head(20).plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("missing rate")
ax.set_title("Top missing columns (train)")
plt.tight_layout()
plt.show()


## 7. Target distribution and skewness

TVT is strictly positive; log1p is only applied when skewness + positivity tests recommend it.


In [ ]:
from pipeline.target_diagnostician import recommend_log1p

y = train_df[target].astype(float)
log_rec = recommend_log1p(y.values)
print(json.dumps(log_rec, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(y.dropna(), bins=40, color="darkorange", edgecolor="white")
axes[0].set_title(f"{target} histogram")
axes[1].boxplot(y.dropna(), vert=True)
axes[1].set_title(f"{target} boxplot")
plt.tight_layout()
plt.show()
print(y.describe())


## 8. TVT_input and prediction zone

`TVT_input` is observed in the known prefix and NaN in the hidden prediction tail — same pattern as top leakage-aware public notebooks.


In [ ]:
if "TVT_input" in train_df.columns:
    known = train_df["TVT_input"].notna().sum()
    print(f"TVT_input known rows: {known:,} / {len(train_df):,} ({100*known/len(train_df):.1f}%)")
    if "MD" in train_df.columns:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.scatter(train_df["MD"], train_df[target], s=4, alpha=0.35, label="TVT")
        ax.scatter(train_df.loc[train_df["TVT_input"].notna(), "MD"],
                   train_df.loc[train_df["TVT_input"].notna(), "TVT_input"],
                   s=4, alpha=0.35, label="TVT_input")
        ax.set_xlabel("MD"); ax.legend(); ax.set_title("TVT vs TVT_input along MD")
        plt.tight_layout(); plt.show()
else:
    print("TVT_input column not present in this extract")


## 9. Well grouping for CV

GroupKFold by well prefix when a stable group key exists; otherwise KFold (see phase 02 recommendations).


In [ ]:
from pipeline import well_group_detector as wgd

group_key = wgd.recommend_group_key(train_df, test_df, id_column=id_col)
groups = wgd.provide_groups(train_df, group_key, id_column=id_col) if group_key else None
print("group_key:", group_key)
if groups is not None:
    print("n_unique_groups:", wgd.count_unique_groups(groups))
    print(groups.value_counts().head())
else:
    print("No well grouping key detected — CV will fall back to KFold")


## 10. Depth coverage (MD)

Check measured depth span — useful for residual-by-depth evaluation in phase 05.


In [ ]:
if "MD" in train_df.columns:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(train_df["MD"].dropna(), bins=50, color="teal", alpha=0.8)
    ax.set_xlabel("MD"); ax.set_title("Measured depth distribution")
    plt.tight_layout(); plt.show()
else:
    print("MD column absent")


## 11. Correlation with target (numeric logs)

Quick screening of log features vs TVT — matches exploratory sections in LightGBM starter kernels.


In [ ]:
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in (target,)]
if num_cols:
    corr = train_df[num_cols + [target]].corr(numeric_only=True)[target].drop(target, errors="ignore")
    corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
    fig, ax = plt.subplots(figsize=(6, max(3, 0.25 * len(corr.head(12)))))
    corr.head(12).plot(kind="barh", ax=ax, color="purple")
    ax.set_title("|corr| with target (top 12)")
    plt.tight_layout(); plt.show()


## 12. Train vs test column alignment

Ensure feature columns match between splits before ColumnTransformer fitting.


In [ ]:
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)
only_train = sorted(train_cols - test_cols)
only_test = sorted(test_cols - train_cols)
print("columns only in train:", only_train[:10])
print("columns only in test:", only_test[:10])


## 13. Submission id pattern

Ids encode `{well_hash}_{row_index}` — used later for aligning test predictions.


In [ ]:
sample_full = pd.read_csv(sample_path)
well_prefix = sample_full[id_col].astype(str).str.split("_").str[0]
print("unique wells in submission:", well_prefix.nunique())
print(well_prefix.value_counts().head())


## 14. Target quantiles and tail risk


In [ ]:
print(y.quantile([0.01, 0.05, 0.5, 0.95, 0.99]))


## 15. Sentinel / negative log values

Rogii CSVs use large negative sentinels for missing logs; preprocessor replaces them with NaN.


In [ ]:
feature_cols = [c for c in train_df.columns if c not in (id_col, target)]
neg_frac = {c: float((train_df[c] < -1e9).mean()) for c in feature_cols if c in train_df.select_dtypes("number").columns}
print({k: v for k, v in sorted(neg_frac.items(), key=lambda x: -x[1])[:8] if v > 0})


## 16. EDA summary table (notebook-local)


In [ ]:
eda_preview = {
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "target_mean": float(y.mean()),
    "target_std": float(y.std()),
    "recommended_log1p": log_rec.get("use_log1p"),
    "group_key": group_key,
}
pd.Series(eda_preview)


## 17. Persist phase artifacts

Write JSON handoff files and `phase_manifest.json` via `phase_runner` (trace-aligned).


In [ ]:
from phase_runner import run_01_data_analysis

manifest = run_01_data_analysis(data_dir=DATA_DIR)
print("Phase manifest:")
for k, v in manifest.items():
    print(f"  {k}: {v}")


## 18. Artifact inspection


In [ ]:
phase_dir = ARTIFACTS_ROOT / PHASE
for p in sorted(phase_dir.iterdir()):
    print(p.name, "→", p.stat().st_size, "bytes")
